# HyperKT — Stage 9B: Checkpoint and Sequence Integrity Audit
**CPU runtime; no training; no test-set predictive evaluation.** Upload the Google Drive checkpoint ZIP, Stage 2 ZIP, Stage 3B ZIP, and optionally Stage 8B results ZIP. Outputs `hyperkt_stage9b.zip`.

This audit loads all 20 checkpoints, recomputes validation predictions and compares saved metrics; checks causal non-interference on *loaded weights*, padding/shift conventions, student disjointness, and repeated target windows. It cannot establish global event uniqueness without original interaction/event IDs. **The held-out test data is not read.**

In [ ]:
import io,os,json,zipfile,hashlib,random,platform,collections
from pathlib import Path
import numpy as np,pandas as pd,torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from sklearn.metrics import roc_auc_score,log_loss,brier_score_loss,f1_score
from google.colab import files
OUT=Path('/content/hyperkt_stage9b');OUT.mkdir(exist_ok=True)
DEVICE='cpu'
print('torch',torch.__version__,'device',DEVICE)

## 1. Upload four archives (results ZIP optional)

In [ ]:
uploaded=files.upload()
archives={name:zipfile.ZipFile(io.BytesIO(blob)) for name,blob in uploaded.items() if name.lower().endswith('.zip')}
def has_suffix(z,suffix):return any(x.endswith(suffix) for x in z.namelist())
def pick(suffix):
    found=[(name,z) for name,z in archives.items() if has_suffix(z,suffix)]
    if len(found)!=1:raise ValueError(f'Expected one ZIP containing {suffix}, found {[x[0] for x in found]}')
    return found[0]
name2,z2=pick('train_sequences.npz');name3,z3=pick('hypergraph_repaired.npz');namecp,zcp=pick('best.pt')
name8,z8=next(((name,z) for name,z in archives.items() if has_suffix(z,'all_results.csv')), (None,None))
print('Stage2:',name2,'Stage3B:',name3,'Checkpoints:',namecp,'Results:',name8)
def read_npz(z,name):
    with np.load(io.BytesIO(z.read(name)),allow_pickle=False) as a:return {k:a[k].copy() for k in a.files}
train=read_npz(z2,'train_sequences.npz');val=read_npz(z2,'val_sequences.npz')
struct=read_npz(z3,'hypergraph_repaired.npz')
N_SKILLS=int(max(train['target_skill'].max(),val['target_skill'].max(),struct['skill_ids'].max())+1)
H=np.zeros((N_SKILLS,struct['assignments'].shape[1]),dtype=np.float32)
H[struct['skill_ids'].astype(int)]=struct['assignments'].astype(np.float32)
MAX_LEN=train['history_skill'].shape[1]
assert N_SKILLS==151 and MAX_LEN==200
CFG=dict(hidden=128,layers=2,heads=8,dropout=.1,batch_size=16,lr=2e-4,weight_decay=1e-4,max_epochs=20,patience=5)
print('Loaded',len(train['mask']),'training and',len(val['mask']),'validation windows')

## 2. Model definition copied from Stage 8B

In [ ]:
class SeqDataset(Dataset):
    def __init__(self,d): self.d=d
    def __len__(self):return len(self.d['mask'])
    def __getitem__(self,i):
        return tuple(torch.as_tensor(self.d[k][i],dtype=torch.long if k in ('history_skill','history_correct','target_skill') else torch.float32) for k in ('history_skill','history_correct','target_skill','target_correct','mask'))
TRAIN_DS=SeqDataset(train); VAL_DS=SeqDataset(val)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
    try:torch.use_deterministic_algorithms(True,warn_only=True)
    except Exception:pass
class CausalBlock(nn.Module):
    def __init__(self,d=128,heads=8,drop=.1,decay=False):
        super().__init__();self.heads=heads;self.dhead=d//heads;self.decay=decay
        self.qkv=nn.Linear(d,3*d);self.proj=nn.Linear(d,d);self.drop=nn.Dropout(drop)
        self.ln1=nn.LayerNorm(d);self.ln2=nn.LayerNorm(d)
        self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Dropout(drop),nn.Linear(4*d,d))
        if decay:self.log_rate=nn.Parameter(torch.full((heads,),-3.0))
    def forward(self,x):
        B,T,D=x.shape; z=self.ln1(x)
        q,k,v=self.qkv(z).chunk(3,dim=-1)
        def split(t):return t.reshape(B,T,self.heads,self.dhead).transpose(1,2)
        q,k,v=map(split,(q,k,v))
        logits=torch.matmul(q,k.transpose(-2,-1))*(self.dhead**-.5)
        pos=torch.arange(T,device=x.device)
        dist=(pos[:,None]-pos[None,:]).clamp(min=0).float()
        if self.decay:logits=logits-torch.nn.functional.softplus(self.log_rate)[None,:,None,None]*dist[None,None,:,:]
        future=pos[None,:]>pos[:,None]
        logits=logits.masked_fill(future[None,None,:,:],torch.finfo(logits.dtype).min)
        a=torch.softmax(logits,dim=-1)
        x=x+self.drop(self.proj(torch.matmul(self.drop(a),v).transpose(1,2).reshape(B,T,D)))
        x=x+self.drop(self.ff(self.ln2(x)))
        return x
class HyperKT(nn.Module):
    def __init__(self,n_skills,H,hyper=False,decay=False,d=128,layers=2,heads=8,drop=.1):
        super().__init__();self.hyper=hyper
        self.skill=nn.Embedding(n_skills,d,padding_idx=1)
        self.response=nn.Embedding(2,d)
        self.pos=nn.Embedding(MAX_LEN,d)
        self.blocks=nn.ModuleList([CausalBlock(d,heads,drop,decay) for _ in range(layers)])
        self.norm=nn.LayerNorm(d)
        if hyper:
            self.register_buffer('assignments',torch.tensor(H))
            self.edge_embedding=nn.Parameter(torch.randn(H.shape[1],d)*.02)
            self.fusion=nn.Linear(d*2,d)
        self.output=nn.Sequential(nn.LayerNorm(d*2),nn.Linear(d*2,d),nn.GELU(),nn.Dropout(drop),nn.Linear(d,1))
    def forward(self,hs,hc,ts):
        B,T=hs.shape; pos=torch.arange(T,device=hs.device)[None,:]
        x=self.skill(hs)+self.response(hc.clamp(0,1))+self.pos(pos)
        for layer in self.blocks:x=layer(x)
        x=self.norm(x)
        target=self.skill(ts)
        if self.hyper:
            structural=self.assignments[ts]@self.edge_embedding
            target=target+torch.tanh(self.fusion(torch.cat([target,structural],dim=-1)))
        return self.output(torch.cat([x,target],dim=-1)).squeeze(-1)
VARIANTS={'transformer':(False,False),'transformer_decay':(False,True),'transformer_hypergraph':(True,False),'transformer_hypergraph_decay':(True,True)}
SEEDS=[42,123,2024,3407,7777]


## 3. Checkpoints: coverage, safe load, state-dict shape and metadata

In [ ]:
ptnames=sorted(x for x in zcp.namelist() if x.endswith('/best.pt'))
expected={f'{v}_seed{s}' for v in VARIANTS for s in SEEDS}
found={x.split('/')[-2] for x in ptnames}
assert len(ptnames)==20 and found==expected,(len(ptnames),sorted(expected-found),sorted(found-expected))
rows=[];loaded={}
for path in ptnames:
    blob=zcp.read(path)
    ck=torch.load(io.BytesIO(blob),map_location='cpu',weights_only=True)
    variant=ck['variant'];seed=int(ck['seed']);hyper,decay=VARIANTS[variant]
    assert path.split('/')[-2]==f'{variant}_seed{seed}'
    assert ck['config']==CFG,(path,ck['config'])
    model=HyperKT(N_SKILLS,H,hyper,decay,CFG['hidden'],CFG['layers'],CFG['heads'],CFG['dropout'])
    model.load_state_dict(ck['state_dict'],strict=True);model.eval()
    finite=all(torch.isfinite(t).all().item() for t in model.state_dict().values() if torch.is_floating_point(t))
    assert finite,path
    loaded[(variant,seed)]=model
    rows.append({'variant':variant,'seed':seed,'best_epoch':int(ck['best_epoch']),'sha256':hashlib.sha256(blob).hexdigest(),'size_bytes':len(blob),'strict_load':True,'finite_tensors':finite})
pd.DataFrame(rows).to_csv(OUT/'checkpoint_inventory.csv',index=False)
print('PASS:',len(rows),'checkpoints loaded strictly, finite weights')

## 4. Sequence integrity, window overlap and target-event limitations

In [ ]:
def check_sequences(d,label):
    m=d['mask'].astype(bool);assert m.ndim==2 and m.shape[1]==MAX_LEN
    assert all(d[k].shape==m.shape for k in ['history_skill','history_correct','target_skill','target_correct'])
    assert d['user_id'].shape==(len(m),)
    assert np.all(~m[:,1:] | m[:,:-1]),'non-prefix mask'
    adj=m[:,1:] & m[:,:-1]
    assert np.array_equal(d['history_skill'][:,1:][adj],d['target_skill'][:,:-1][adj])
    assert np.array_equal(d['history_correct'][:,1:][adj],d['target_correct'][:,:-1][adj])
    assert np.isin(d['target_correct'][m],[0,1]).all()
    assert (d['target_skill'][m]>=2).all()
    assert (d['target_skill'][m]<N_SKILLS).all()
    assert np.isin(d['history_correct'][m],[0,1]).all()
    # Detect identical full target sequences for same student, not identical individual events.
    hashes=[]
    for i in range(len(m)):
        valid=m[i];b=d['target_skill'][i,valid].tobytes()+d['target_correct'][i,valid].tobytes()
        hashes.append((str(d['user_id'][i]),hashlib.sha256(b).hexdigest()))
    dup=len(hashes)-len(set(hashes))
    return {'split':label,'windows':len(m),'students':len(set(d['user_id'])),'valid_targets':int(m.sum()),'duplicate_identical_windows':dup,'mask_shift_checks':'pass'}
checks=[check_sequences(train,'train'),check_sequences(val,'validation')]
assert not(set(train['user_id'])&set(val['user_id']))
pd.DataFrame(checks).to_csv(OUT/'sequence_audit.csv',index=False)
print(pd.DataFrame(checks).to_string(index=False))
print('NOTE: Without event IDs, repeated identical skill/response values cannot prove or disprove target-event duplication across overlapping windows.')

## 5. Causal non-interference on every recovered checkpoint; padding invariance on valid prefix

In [ ]:
causal=[]
with torch.no_grad():
    valid_indices=np.where(val['mask'].sum(axis=1)>=100)[0]
    assert len(valid_indices)>0
    idx=int(valid_indices[0]);cut=50
    hs=torch.tensor(val['history_skill'][idx:idx+1],dtype=torch.long)
    hc=torch.tensor(val['history_correct'][idx:idx+1],dtype=torch.long)
    ts=torch.tensor(val['target_skill'][idx:idx+1],dtype=torch.long)
    hs2=hs.clone();hc2=hc.clone();ts2=ts.clone()
    hs2[:,cut:]=2;hc2[:,cut:]=1-hc2[:,cut:];ts2[:,cut:]=2
    for (variant,seed),model in loaded.items():
        a=model(hs,hc,ts);b=model(hs2,hc2,ts2)
        delta=float((a[:,:cut]-b[:,:cut]).abs().max())
        assert delta<1e-5,(variant,seed,delta)
        causal.append({'variant':variant,'seed':seed,'max_earlier_logit_delta':delta,'pass':True})
pd.DataFrame(causal).to_csv(OUT/'causal_checks.csv',index=False)
print('PASS: 20 loaded checkpoints satisfy future-interaction non-interference; max delta',max(r['max_earlier_logit_delta'] for r in causal))
print('NOTE: Stage 8B attention has no explicit key-padding mask; prefix-valid windows + causal mask prevent right-padding from affecting valid positions. Do not infer that arbitrary interior padding is safe.')

## 6. Recompute validation metrics from saved checkpoints (20 runs; CPU can take several minutes)

In [ ]:
def metric(y,p):
    y=np.asarray(y,dtype=np.int64);p=np.clip(np.asarray(p,dtype=np.float64),1e-7,1-1e-7)
    edges=np.linspace(0,1,16);inds=np.minimum(np.digitize(p,edges[1:-1]),14)
    ece=sum((inds==i).mean()*abs(y[inds==i].mean()-p[inds==i].mean()) for i in range(15) if np.any(inds==i))
    return dict(auc=float(roc_auc_score(y,p)),nll=float(log_loss(y,p,labels=[0,1])),brier=float(brier_score_loss(y,p)),f1=float(f1_score(y,p>=.5,zero_division=0)),ece15=float(ece),n=int(len(y)))
# Uses the exact evaluation flattening convention of Stage 8B.
@torch.no_grad()
def evaluate_checkpoint(model):
    ys=[];ps=[]
    for start in range(0,len(val['mask']),CFG['batch_size']):
        stop=start+CFG['batch_size']
        hs=torch.as_tensor(val['history_skill'][start:stop],dtype=torch.long)
        hc=torch.as_tensor(val['history_correct'][start:stop],dtype=torch.long)
        ts=torch.as_tensor(val['target_skill'][start:stop],dtype=torch.long)
        mask=val['mask'][start:stop].astype(bool)
        p=torch.sigmoid(model(hs,hc,ts)).numpy()
        ys.append(val['target_correct'][start:stop][mask]);ps.append(p[mask])
    return metric(np.concatenate(ys),np.concatenate(ps))
validation=[]
for (variant,seed),model in loaded.items():
    stats=evaluate_checkpoint(model)
    validation.append({'variant':variant,'seed':seed,**stats})
    print(f'{variant} seed {seed}: AUC={stats["auc"]:.6f} NLL={stats["nll"]:.6f}',flush=True)
recomputed=pd.DataFrame(validation).sort_values(['variant','seed'])
recomputed.to_csv(OUT/'recomputed_validation_metrics.csv',index=False)
print('Recomputed all 20 validation runs')

## 7. Compare recomputed metrics against original saved best_metrics.json

In [ ]:
comparisons=[]
for r in validation:
    variant,seed=r['variant'],r['seed']
    path=next(x for x in zcp.namelist() if x.endswith(f'/{variant}_seed{seed}/best_metrics.json'))
    saved=json.loads(zcp.read(path))
    deltas={f'delta_{k}':float(r[k]-saved[k]) for k in ('auc','nll','brier','f1','ece15')}
    comparisons.append({'variant':variant,'seed':seed,**deltas,'saved_n':int(saved['n']),'recomputed_n':int(r['n'])})
diffs=pd.DataFrame(comparisons);diffs.to_csv(OUT/'saved_vs_recomputed.csv',index=False)
maxdiff=float(diffs[[c for c in diffs if c.startswith('delta_')]].abs().max().max())
assert (diffs['saved_n']==diffs['recomputed_n']).all()
print('Maximum absolute metric difference:',maxdiff)
print('Reproducibility tolerance 1e-5:', 'PASS' if maxdiff<1e-5 else 'REVIEW REQUIRED')
if maxdiff>=1e-5:print(diffs.to_string(index=False))

## 8. Audit verdict and export

In [ ]:
report={'stage':'9B','checkpoints':len(loaded),'checkpoint_load':'pass','causal_probe':'pass','student_disjoint_train_val':True,'sequence_shift':'pass','duplicate_identical_windows':{r['split']:r['duplicate_identical_windows'] for r in checks},'cross_window_target_event_uniqueness':'unresolved: original event IDs not present','test_predictions_evaluated':False,'max_saved_vs_recomputed_metric_delta':maxdiff,'metric_reproduction_pass':bool(maxdiff<1e-5),'stage8b_config':CFG,'limitations':['Fixed pretrained Stage 3B hypergraph across seeds','No explicit key padding mask; only right-padded prefix-valid windows are supported','Positional distance is not elapsed clock time','Cannot prove unique event identities from skill/response pairs']}
(OUT/'stage9b_report.json').write_text(json.dumps(report,indent=2))
(OUT/'README.txt').write_text('Stage 9B: checkpoint loading, causal noninterference and validation reproducibility. No test evaluation. Cross-window event uniqueness remains unresolved without original event IDs.')
archive='/content/hyperkt_stage9b.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in OUT.iterdir():
        if f.is_file():z.write(f,f.name)
print(json.dumps(report,indent=2));print('Created',archive)
files.download(archive)